In [1]:
#kernel thesis clean4
import pickle
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import math
import torch.nn.functional as F

In [2]:
df = pd.read_pickle("C:\\Users\\Patrick\\Masterthesis\\Benchmarks\\screw_data_s02-v2_identical-to-v1.pkl")
df.head()

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [3]:
x_data = np.array(df['torque_values'].tolist())[..., np.newaxis]  
y_data = np.array(df['class_values'].tolist())
le = LabelEncoder()
y_encoded = le.fit_transform(y_data)
angle = np.array(df['angle_values'].tolist())[..., np.newaxis]
phase = np.array(df['step_values'].tolist())[..., np.newaxis]

angle = np.transpose(angle, (0, 2, 1))
phase = np.transpose(phase, (0, 2, 1))
x_data = np.transpose(x_data, (0, 2, 1))  #reshaped to (num_samples, num_features, sequence_length)

X_train_full, X_test, angle_train_full, angle_test, phase_train_full, phase_test, y_train_full, y_test = train_test_split(x_data, angle, phase, y_encoded, test_size=0.2, stratify=y_encoded,random_state=42)
X_train, X_val, angle_train, angle_val, phase_train, phase_val, y_train, y_val = train_test_split(X_train_full, angle_train_full, phase_train_full, y_train_full,test_size=0.25,stratify=y_train_full,random_state=42) #0.25 * 0.8 = 0.2

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

#angle und phase tensoren
angle_train = torch.tensor(angle_train, dtype=torch.float32)
angle_val   = torch.tensor(angle_val, dtype=torch.float32)
angle_test  = torch.tensor(angle_test, dtype=torch.float32)

phase_train = torch.tensor(phase_train, dtype=torch.float32)
phase_val   = torch.tensor(phase_val, dtype=torch.float32)
phase_test  = torch.tensor(phase_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, angle_train, phase_train, y_train)
val_dataset   = TensorDataset(X_val, angle_val, phase_val, y_val)
test_dataset  = TensorDataset(X_test, angle_test, phase_test, y_test)



train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", X_train.shape, angle_train.shape, phase_train.shape, y_train.shape)
print("Validation:", X_val.shape, angle_val.shape, phase_val.shape, y_val.shape)
print("Test:", X_test.shape, angle_test.shape, phase_test.shape, y_test.shape)

Train: torch.Size([7500, 1, 800]) torch.Size([7500, 1, 800]) torch.Size([7500, 1, 800]) torch.Size([7500])
Validation: torch.Size([2500, 1, 800]) torch.Size([2500, 1, 800]) torch.Size([2500, 1, 800]) torch.Size([2500])
Test: torch.Size([2500, 1, 800]) torch.Size([2500, 1, 800]) torch.Size([2500, 1, 800]) torch.Size([2500])


In [4]:
class EarlyStopper:
    def __init__(self, patience=1, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [5]:
import torch
import torch.nn as nn
import sys
sys.path.append(r"C:\Users\Patrick\InceptionTime-Pytorch")
from inception import InceptionBlock


class Flatten(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x.mean(-1)   # safer als view


model = nn.Sequential(
    InceptionBlock(
        in_channels=1,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)

M = M_Gewinde + M_Kopf + M_reibung \
Phasen: finding, threadd-forming, pre-tightening und final tightening \
Phase 0: finding = M_reibung \
Phase 1: thread-forming = M_Gewinde + M_reibung \
Phase 2: pre-tightening = M ~ M_elastisch + M_reibung \
Phase 3: final tightening = M_Gewinde + M_kopf + M_reibung

M_Gewinde = F_V * d_2/2 * tan(Steigungswinkel + Gewindereibwinkel) \
M_Kopf = mu_K * F_V * D_K_R/2 \
M_Reibung = M_Gewinde + M_Kopf \
M_elastisch = Steigung von torque-angle * angle # steigung von torque-angle = np.gradient(torque, angle) * angle \

Pre-tightening - spezialfall: M ~ k0 * angle + M_Reibung # Spezialfall da linear-elastischer bereich -> hookesche gesetz \
k0 = dM / dangle \
hookesche gesetz = F = K*x oder F_V = k_axial * delta_längenänderung (einschraubtiefe) \



# PINN Loss - Phase 1: Thread Forming
Formel: M = F_V * d_2/2 * tan(Steigungswinkel + gewindereibwert)
M=Mform​+MG​

In [ ]:
def thread_forming_torque(F_V, d2, steigungswinkel, gewindereibwinkel):
    M_thread = F_V * ((d2/2) * torch.tan(steigungswinkel + gewindereibwinkel))
    return M_thread

# PINN Loss- Phase 2: Pre tightening
https://www.leifiphysik.de/mechanik/kraft-und-das-gesetz-von-hooke/grundwissen/gesetz-von-hooke 

F = k_stiffness * deltax

In [ ]:
def pre_tightening_torque(angle,pitch,d2,k_stiffness,screw_length, steigungswinkel, gewindereibwinkel):
    # hookesche gesetz F = k * x, x = pitch * angle / 360
    x_ideal = (pitch * angle) / 360.0  

    x = torch.minimum(x_ideal, screw_length)
    F = k_stiffness * x

    torque_pred = F * (d2 / 2.0) * torch.tan(steigungswinkel + gewindereibwinkel)

    return torque_pred

# PINN Loss - Phase 3: Final Tightening (Anziehdrehmoment)
M = F_V * ((d2 / 2) * torch.tan(steigungswinkel + gewindereibwinkel) + mu_K * d_K_R)


->Final tighteningM=FV(θ) * (d2/2 * tan(s+g) + muk * dkr)


In [ ]:
def final_tightening_torque(F_V, d2, d_K_R, steigungswinkel, gewindereibwinkel, mu_K):
    "Anziehdrehmoment basiert auf M = MG + MK vdi 2230 decomposition"
    M_G = (F_V* (d2 / 2.0)* torch.tan(steigungswinkel + gewindereibwinkel))


    # head friction
    M_K = (mu_K* F_V* d_K_R)

    M_final = M_G + M_K
    return M_final

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Torque_phasen(nn.Module):
    def __init__(self):
        super().__init__()

        self.pitch = torch.tensor(0.00146, dtype=torch.float32)
        self.D = torch.tensor(0.004, dtype=torch.float32)
        self.d2 = self.D - 0.6495 * self.pitch
        self.d3 = torch.tensor(0.00281, dtype=torch.float32)
        self.d_K_R = torch.tensor(0.009 + 0.004 / 2, dtype=torch.float32)

        self.screw_length = torch.tensor(0.0012, dtype=torch.float32)
        self.zul_spannung = torch.tensor(0.9 * 900e6, dtype=torch.float32)

        self.A_S = (torch.pi / 4) * ((self.d2 + self.d3) / 2) ** 2

        self.mu_G = nn.Parameter(torch.tensor([0.12, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24, 0.26]))
        self.mu_K = nn.Parameter(torch.tensor([0.12, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24, 0.26]))

        self.k_stiffness = nn.Parameter(torch.tensor([0.1, 0.1, 0.1, 0.1]))

    def compute_angles(self, device, mu_G):
        beta = torch.deg2rad(torch.tensor(20.0, device=device))
        steigungswinkel = torch.atan(self.pitch / (torch.pi * self.d2))
        gewindereibwinkel = torch.atan(mu_G / torch.cos(beta / 2))
        return steigungswinkel, gewindereibwinkel

    def build_force(self, angle, k_stiffness):
        x = (self.pitch * angle) / 360.0
        x = torch.minimum(x, self.screw_length.to(angle.device))
        return k_stiffness * x

    def thread_forming_torque(self, Force, d2, steigungswinkel, gewindereibwinkel):
        return Force * (d2 / 2.0) * torch.tan(steigungswinkel + gewindereibwinkel)

    def pre_tightening_torque(self, Force, d2, steigungswinkel, gewindereibwinkel):
        return Force * (d2 / 2.0) * torch.tan(steigungswinkel + gewindereibwinkel)

  
    def final_tightening_torque(self, Force, d2, d_K_R,steigungswinkel, gewindereibwinkel, mu_K):
        M_G = Force * (d2 / 2.0) * torch.tan(steigungswinkel + gewindereibwinkel)
        M_K = Force * mu_K * d_K_R
        return M_G + M_K

    # -------------------------
    def forward(self, phase_mask, angle, X_batch, class_input):

        device = X_batch.device

        mu_G = self.mu_G[class_input].to(device)[:, None, None]
        mu_K = self.mu_K[class_input].to(device)[:, None, None]
        d2 = self.d2.to(device)
        d_K_R = self.d_K_R.to(device)

        steigungswinkel, gewindereibwinkel = self.compute_angles(device, mu_G)

        # force (Hooke)
        k = self.k_stiffness[phase_mask.long()]  #phasen abhängig
        Force = self.build_force(angle, k)

        M_thread = self.thread_forming_torque(Force, d2, steigungswinkel, gewindereibwinkel)
        M_pre = self.pre_tightening_torque(Force, d2, steigungswinkel, gewindereibwinkel)

        M_final = self.final_tightening_torque(Force, d2, d_K_R,steigungswinkel, gewindereibwinkel, mu_K)

        torque_phys = ((phase_mask == 1) * M_thread +(phase_mask == 2) * M_pre +(phase_mask == 3) * M_final)
        loss = F.mse_loss(torque_phys, X_batch)
        return loss

In [ ]:
#X_train, angle_train, phase_train, y_train)

In [12]:
from sklearn.metrics import f1_score
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
physics_loss = Torque_phasen().to(device)
optimizer = torch.optim.AdamW(list(model.parameters()) + list(physics_loss.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

earlystop = EarlyStopper(patience=20, min_delta=0.001)

epochs = 50

train_losses = []
train_physics_losses = []
val_losses = []
val_physics_losses = []
val_f1_scores = []

best_val_f1 = -np.inf
best_model_state = None
lambda_phys = 1.0

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    total_phys = 0.0

    for X_batch, angle_batch, phase_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        angle_batch = angle_batch.to(device)
        phase_batch = phase_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss_class = criterion(outputs, y_batch)
        
        #loss phyisics
        loss_physics = physics_loss(phase_batch, angle_batch, X_batch, y_batch)
        loss = loss_class + lambda_phys * loss_physics
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_phys += loss_physics.item()

    avg_train_loss = total_loss / len(train_loader)
    avg_train_phys_loss = total_phys / len(train_loader)
    train_losses.append(avg_train_loss)
    train_physics_losses.append(avg_train_phys_loss)
    model.eval()

    val_loss = 0.0
    val_physics_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_val_batch, angle_val_batch, phase_val_batch, y_val_batch in val_loader:
            X_val_batch = X_val_batch.to(device)
            angle_val_batch = angle_val_batch.to(device)
            phase_val_batch = phase_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            outputs = model(X_val_batch)
            loss_class = criterion(outputs, y_val_batch)
            loss_physics = physics_loss(phase_val_batch, angle_val_batch, X_val_batch, y_val_batch)
            loss = loss_class + lambda_phys * loss_physics

            val_loss += loss.item()
            val_physics_loss += lambda_phys * loss_physics.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_val_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    avg_val_phys_loss = val_physics_loss / len(val_loader)
    val_physics_losses.append(avg_val_phys_loss)

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_f1_scores.append(val_f1)
    scheduler.step(avg_val_loss)

    #best model speichern für test
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()

    if earlystop.early_stop(avg_val_loss):
        print("Early stopping triggered")
        break

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Train Physics Loss: {train_physics_losses[-1]:.4f}, Val Physics Loss: {val_physics_losses[-1]:.4f}, Val F1 Score: {val_f1:.4f}")


Epoch 1/50, Train Loss: 1.2954, Val Loss: 2.0101, Train Physics Loss: 0.1621, Val Physics Loss: 0.1642, Val F1 Score: 0.3265
Epoch 2/50, Train Loss: 1.2581, Val Loss: 1.7991, Train Physics Loss: 0.1623, Val Physics Loss: 0.1642, Val F1 Score: 0.2880
Epoch 3/50, Train Loss: 1.2384, Val Loss: 3.5801, Train Physics Loss: 0.1622, Val Physics Loss: 0.1642, Val F1 Score: 0.2512
Epoch 4/50, Train Loss: 1.2450, Val Loss: 1.6549, Train Physics Loss: 0.1622, Val Physics Loss: 0.1642, Val F1 Score: 0.3727
Epoch 5/50, Train Loss: 1.2330, Val Loss: 1.4219, Train Physics Loss: 0.1621, Val Physics Loss: 0.1642, Val F1 Score: 0.4041
Epoch 6/50, Train Loss: 1.2355, Val Loss: 1.6461, Train Physics Loss: 0.1622, Val Physics Loss: 0.1642, Val F1 Score: 0.3992
Epoch 7/50, Train Loss: 1.2256, Val Loss: 2.1468, Train Physics Loss: 0.1622, Val Physics Loss: 0.1641, Val F1 Score: 0.2703
Epoch 8/50, Train Loss: 1.2129, Val Loss: 1.8954, Train Physics Loss: 0.1621, Val Physics Loss: 0.1641, Val F1 Score: 0.3440


In [13]:
model.load_state_dict(best_model_state)
#torch.save(best_model_state, "best_inception_model.pth")
model.eval()

test_preds = []
test_labels = []
test_loss = 0.0

with torch.no_grad():
    for X_test_batch, phase_test_batch, angle_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        angle_test_batch = angle_test_batch.to(device)
        phase_test_batch = phase_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        outputs = model(X_test_batch)
        loss = criterion(outputs, y_test_batch)

        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

Test Loss: 0.9930
Test F1 Macro: 0.5530


In [14]:
physics_loss.mu_G

Parameter containing:
tensor([5.2490, 5.3068, 4.7340, 4.6016, 4.7682, 4.7586, 4.8663, 4.8580],
       device='cuda:0', requires_grad=True)

In [15]:
physics_loss.mu_K

Parameter containing:
tensor([4.5712, 4.6152, 4.2639, 4.0344, 4.2471, 4.2025, 4.3504, 4.3331],
       device='cuda:0', requires_grad=True)

In [16]:
physics_loss.k_stiffness

Parameter containing:
tensor([0.1000, 5.1027, 5.0729, 4.9538], device='cuda:0', requires_grad=True)

# Parameter eingesetzt und auf logits gerechnet:

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Torque_phasen(nn.Module):
    def __init__(self):
        super().__init__()

        self.pitch = torch.tensor(0.00146, dtype=torch.float32)
        self.D = torch.tensor(0.004, dtype=torch.float32)
        self.d2 = self.D - 0.6495 * self.pitch
        self.d3 = torch.tensor(0.00281, dtype=torch.float32)
        self.d_K_R = torch.tensor(0.009 + 0.004 / 2, dtype=torch.float32)

        self.screw_length = torch.tensor(0.0012, dtype=torch.float32)
        self.zul_spannung = torch.tensor(0.9 * 900e6, dtype=torch.float32)

        self.A_S = (torch.pi / 4) * ((self.d2 + self.d3) / 2) ** 2

        #self.mu_G = nn.Parameter(torch.tensor([0.12, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24, 0.26]))
        #self.mu_K = nn.Parameter(torch.tensor([0.12, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24, 0.26]))
        self.mu_G = torch.tensor([5.2490, 5.3068, 4.7340, 4.6016, 4.7682, 4.7586, 4.8663, 4.8580], dtype=torch.float32)
        self.mu_K = torch.tensor([4.5712, 4.6152, 4.2639, 4.0344, 4.2471, 4.2025, 4.3504, 4.3331], dtype=torch.float32)
        self.k_stiffness = torch.tensor([0.1000, 5.1027, 5.0729, 4.9538], dtype=torch.float32)
        #self.k_stiffness = nn.Parameter(torch.tensor([0.1, 0.1, 0.1, 0.1]))

    def compute_angles(self, device, mu_G):
        beta = torch.deg2rad(torch.tensor(20.0, device=device))
        steigungswinkel = torch.atan(self.pitch / (torch.pi * self.d2))
        gewindereibwinkel = torch.atan(mu_G / torch.cos(beta / 2))
        return steigungswinkel, gewindereibwinkel

    def build_force(self, angle, k_stiffness):
        x = (self.pitch * angle) / 360.0
        x = torch.minimum(x, self.screw_length.to(angle.device))
        return k_stiffness * x

    def thread_forming_torque(self, Force, d2, steigungswinkel, gewindereibwinkel):
        return Force * (d2 / 2.0) * torch.tan(steigungswinkel + gewindereibwinkel)

    def pre_tightening_torque(self, Force, d2, steigungswinkel, gewindereibwinkel):
        return Force * (d2 / 2.0) * torch.tan(steigungswinkel + gewindereibwinkel)

  
    def final_tightening_torque(self, Force, d2, d_K_R,steigungswinkel, gewindereibwinkel, mu_K):
        M_G = Force * (d2 / 2.0) * torch.tan(steigungswinkel + gewindereibwinkel)
        M_K = Force * mu_K * d_K_R
        return M_G + M_K

    def forward(self, logits, phase_mask, angle, X_batch, class_input):

        device = X_batch.device
        probs = torch.softmax(logits, dim=-1)

        #mu_G = self.mu_G[class_input].to(device)[:, None, None]
        #mu_K = self.mu_K[class_input].to(device)[:, None, None]
        mu_K = self.mu_K.to(device)
        mu_G = self.mu_G.to(device)
        mu_K = torch.sum(probs * mu_K, dim=-1)
        mu_G = torch.sum(probs * mu_G, dim=-1)
        mu_K = mu_K[:, None, None]
        mu_G = mu_G[:, None, None]
        d2 = self.d2.to(device)
        d_K_R = self.d_K_R.to(device)

        steigungswinkel, gewindereibwinkel = self.compute_angles(device, mu_G)

        # force (Hooke formel)
        k = self.k_stiffness.to(device)[phase_mask.long()]  #phasen abhängig
        Force = self.build_force(angle, k)

        M_thread = self.thread_forming_torque(Force, d2, steigungswinkel, gewindereibwinkel)
        M_pre = self.pre_tightening_torque(Force, d2, steigungswinkel, gewindereibwinkel)

        M_final = self.final_tightening_torque(Force, d2, d_K_R,steigungswinkel, gewindereibwinkel, mu_K)

        torque_phys = ((phase_mask == 1) * M_thread +(phase_mask == 2) * M_pre +(phase_mask == 3) * M_final)
        loss = F.mse_loss(torque_phys, X_batch)
        return loss

In [8]:
from sklearn.metrics import f1_score
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Sequential(
    InceptionBlock(
        in_channels=1,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    InceptionBlock(
        in_channels=32 * 4,
        n_filters=32,
        kernel_sizes=[5, 11, 23],
        bottleneck_channels=32,
        use_residual=True
    ),
    nn.AdaptiveAvgPool1d(1),
    Flatten(),
    nn.Linear(32 * 4, 8)   # 8 Klassen
)
model.to(device)

criterion = nn.CrossEntropyLoss()
physics_loss = Torque_phasen().to(device)
optimizer = torch.optim.AdamW(list(model.parameters()) + list(physics_loss.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

earlystop = EarlyStopper(patience=20, min_delta=0.001)

epochs = 50

train_losses = []
train_physics_losses = []
val_losses = []
val_physics_losses = []
val_f1_scores = []

best_val_f1 = -np.inf
best_model_state = None
lambda_phys = 1.0

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    total_phys = 0.0

    for X_batch, angle_batch, phase_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        angle_batch = angle_batch.to(device)
        phase_batch = phase_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss_class = criterion(outputs, y_batch)
        
        #loss phyisics
        loss_physics = physics_loss(outputs, phase_batch, angle_batch, X_batch, y_batch)
        loss = loss_class + lambda_phys * loss_physics
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_phys += loss_physics.item()

    avg_train_loss = total_loss / len(train_loader)
    avg_train_phys_loss = total_phys / len(train_loader)
    train_losses.append(avg_train_loss)
    train_physics_losses.append(avg_train_phys_loss)
    model.eval()

    val_loss = 0.0
    val_physics_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_val_batch, angle_val_batch, phase_val_batch, y_val_batch in val_loader:
            X_val_batch = X_val_batch.to(device)
            angle_val_batch = angle_val_batch.to(device)
            phase_val_batch = phase_val_batch.to(device)
            y_val_batch = y_val_batch.to(device)

            outputs = model(X_val_batch)
            loss_class = criterion(outputs, y_val_batch)
            loss_physics = physics_loss(outputs, phase_val_batch, angle_val_batch, X_val_batch, y_val_batch)
            loss = loss_class + lambda_phys * loss_physics

            val_loss += loss.item()
            val_physics_loss += lambda_phys * loss_physics.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_val_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    avg_val_phys_loss = val_physics_loss / len(val_loader)
    val_physics_losses.append(avg_val_phys_loss)

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_f1_scores.append(val_f1)
    scheduler.step(avg_val_loss)

    #best model speichern für test
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict()

    if earlystop.early_stop(avg_val_loss):
        print("Early stopping triggered")
        break

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Train Physics Loss: {train_physics_losses[-1]:.4f}, Val Physics Loss: {val_physics_losses[-1]:.4f}, Val F1 Score: {val_f1:.4f}")


Epoch 1/50, Train Loss: 1.9714, Val Loss: 2.2049, Train Physics Loss: 0.1620, Val Physics Loss: 0.1640, Val F1 Score: 0.1453
Epoch 2/50, Train Loss: 1.8268, Val Loss: 3.5595, Train Physics Loss: 0.1620, Val Physics Loss: 0.1640, Val F1 Score: 0.1189
Epoch 3/50, Train Loss: 1.7476, Val Loss: 3.9548, Train Physics Loss: 0.1621, Val Physics Loss: 0.1640, Val F1 Score: 0.1274
Epoch 4/50, Train Loss: 1.7016, Val Loss: 5.1365, Train Physics Loss: 0.1620, Val Physics Loss: 0.1640, Val F1 Score: 0.0618
Epoch 5/50, Train Loss: 1.6975, Val Loss: 1.7031, Train Physics Loss: 0.1620, Val Physics Loss: 0.1640, Val F1 Score: 0.3388
Epoch 6/50, Train Loss: 1.6610, Val Loss: 1.7808, Train Physics Loss: 0.1619, Val Physics Loss: 0.1640, Val F1 Score: 0.2421
Epoch 7/50, Train Loss: 1.6442, Val Loss: 1.8188, Train Physics Loss: 0.1621, Val Physics Loss: 0.1640, Val F1 Score: 0.2683
Epoch 8/50, Train Loss: 1.6285, Val Loss: 3.5419, Train Physics Loss: 0.1619, Val Physics Loss: 0.1640, Val F1 Score: 0.1621


In [9]:
model.load_state_dict(best_model_state)
#torch.save(best_model_state, "best_inception_model.pth")
model.eval()

test_preds = []
test_labels = []
test_loss = 0.0

with torch.no_grad():
    for X_test_batch, phase_test_batch, angle_test_batch, y_test_batch in test_loader:
        X_test_batch = X_test_batch.to(device)
        angle_test_batch = angle_test_batch.to(device)
        phase_test_batch = phase_test_batch.to(device)
        y_test_batch = y_test_batch.to(device)

        outputs = model(X_test_batch)
        loss = criterion(outputs, y_test_batch)

        test_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(y_test_batch.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_f1 = f1_score(test_labels, test_preds, average="macro")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

Test Loss: 1.0855
Test F1 Macro: 0.5017
